# 03 — Statistical Testing on Cleaned Data

Which predictors are associated with 30-day readmission?

This notebook uses `data/processed/cleaned.csv`, which is produced by `01_cleaning.ipynb`. Statistical tests are performed after data cleaning so impossible clinical values do not affect group comparisons.

- categorical: chi-square test of independence with Cramér's V effect size
- numeric: Welch t-test, or Mann-Whitney U when the feature is very skewed, with mean differences and Cohen's d
- multiple testing: Benjamini-Hochberg correction because several tests are run
- permutation test: difference in mean adherence score by readmission group


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests

from src.config import PROCESSED_DIR, TARGET, FIGURES_DIR, TABLES_DIR

In [ ]:
cleaned_path = PROCESSED_DIR / 'cleaned.csv'

if not cleaned_path.exists():
    raise FileNotFoundError(
        f'{cleaned_path} was not found. Run 01_cleaning.ipynb first to create cleaned.csv.'
    )

df = pd.read_csv(cleaned_path)
y = df[TARGET]

num = df.select_dtypes('number').drop(columns=[TARGET]).columns.tolist()
binary = [c for c in num if df[c].dropna().isin([0, 1]).all()]
num = [c for c in num if c not in binary]
cat = df.select_dtypes(exclude='number').columns.tolist() + binary

print('Rows, columns:', df.shape)
print('numeric:', num)
print('categorical:', cat)


In [ ]:
rows = []

for c in cat:
    table = pd.crosstab(df[c], y)
    chi2, p, dof, expected = stats.chi2_contingency(table)

    n = table.to_numpy().sum()
    r, k = table.shape
    denom = n * (min(r, k) - 1)
    cramers_v = np.sqrt(chi2 / denom) if denom > 0 else np.nan

    rows.append({
        'feature': c,
        'chi2': chi2,
        'dof': dof,
        'cramers_v': cramers_v,
        'p_raw': p
    })

chi = pd.DataFrame(rows)
chi['p_adj'] = multipletests(chi['p_raw'], method='fdr_bh')[1]
chi['significant'] = chi['p_adj'] < 0.05
chi = chi.sort_values('p_adj')

chi.to_csv(TABLES_DIR / 'tbl_categorical_vs_target.csv', index=False)
chi.round(4)


In [ ]:
def cohens_d(a, b):
    """Calculate Cohen's d using pooled standard deviation."""
    n1, n2 = len(a), len(b)
    if n1 < 2 or n2 < 2:
        return np.nan

    s1, s2 = a.std(ddof=1), b.std(ddof=1)
    pooled_sd = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))

    if pooled_sd == 0:
        return np.nan

    return (a.mean() - b.mean()) / pooled_sd


rows = []

for c in num:
    a = df.loc[y == 1, c].dropna()
    b = df.loc[y == 0, c].dropna()
    feature_skew = stats.skew(df[c].dropna())

    if abs(feature_skew) > 1:
        stat, p = stats.mannwhitneyu(a, b)
        test = 'mannwhitney'
    else:
        stat, p = stats.ttest_ind(a, b, equal_var=False)
        test = 'welch_t'

    rows.append({
        'feature': c,
        'test': test,
        'skew': feature_skew,
        'mean_readmit': a.mean(),
        'mean_no': b.mean(),
        'mean_difference': a.mean() - b.mean(),
        'cohens_d': cohens_d(a, b),
        'stat': stat,
        'p_raw': p
    })

numres = pd.DataFrame(rows)
numres['p_adj'] = multipletests(numres['p_raw'], method='fdr_bh')[1]
numres['significant'] = numres['p_adj'] < 0.05
numres = numres.sort_values('p_adj')

numres.to_csv(TABLES_DIR / 'tbl_numeric_vs_target.csv', index=False)
numres.round(4)


In [ ]:
# 95% confidence interval for each numeric mean
rows = []
for c in num:
    x = df[c].dropna()
    se = x.std(ddof=1) / np.sqrt(len(x))
    lo, hi = stats.t.interval(0.95, len(x) - 1, loc=x.mean(), scale=se)
    rows.append({'feature': c, 'mean': x.mean(), 'ci_low': lo, 'ci_high': hi})
cis = pd.DataFrame(rows)
cis.to_csv(TABLES_DIR / 'tbl_mean_cis.csv', index=False)
cis.round(3)

In [ ]:
# standardized residuals from the ace_inhibitor vs target table
table = pd.crosstab(df['ace_inhibitor'], y)
chi2, p, dof, expected = stats.chi2_contingency(table)
resid = (table - expected) / np.sqrt(expected)
resid.index = [f'ace={i}' for i in resid.index]
resid.columns = [f'readmit={c}' for c in resid.columns]
resid.to_csv(TABLES_DIR / 'tbl_std_residuals_ace.csv')
resid.round(3)

In [ ]:
# univariate odds ratios for the binary predictors
ors = []
for c in ['ace_inhibitor', 'beta_blocker', 'diuretic']:
    m = sm.Logit(y, sm.add_constant(df[c])).fit(disp=0)
    lo, hi = np.exp(m.conf_int().loc[c])
    ors.append({'feature': c, 'OR': np.exp(m.params[c]), 'lo': lo, 'hi': hi})
g = (df['gender'] == 'Male').astype(int).rename('gender_male')
m = sm.Logit(y, sm.add_constant(g)).fit(disp=0)
lo, hi = np.exp(m.conf_int().loc['gender_male'])
ors.append({'feature': 'gender_male', 'OR': np.exp(m.params['gender_male']), 'lo': lo, 'hi': hi})
ors = pd.DataFrame(ors)
pos = np.arange(len(ors))
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(ors['OR'], pos, xerr=[ors['OR'] - ors['lo'], ors['hi'] - ors['OR']], fmt='o')
ax.axvline(1, color='grey', ls='--')
ax.set_yticks(pos)
ax.set_yticklabels(ors['feature'])
ax.set_xlabel('odds ratio (readmission)')
ax.set_title('Univariate odds ratios')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_odds_ratios.png', dpi=120)
ors.round(3)

In [ ]:
# permutation test: difference in mean adherence by readmission
# Drop missing values first so the test is safe if the dataset changes later.
perm_df = df[[TARGET, 'adherence_score']].dropna()

obs = (
    perm_df.loc[perm_df[TARGET] == 1, 'adherence_score'].mean()
    - perm_df.loc[perm_df[TARGET] == 0, 'adherence_score'].mean()
)

vals = perm_df['adherence_score'].values
labels = perm_df[TARGET].values
rng = np.random.default_rng(0)

perm = []
for _ in range(5000):
    shuffled = rng.permutation(labels)
    diff = vals[shuffled == 1].mean() - vals[shuffled == 0].mean()
    perm.append(diff)

perm = np.array(perm)
pval = np.mean(np.abs(perm) >= abs(obs))

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(perm, bins=40, color='steelblue', alpha=0.8)
ax.axvline(obs, color='red', lw=2, label=f'observed = {obs:.3f}')
ax.set_title(f'Permutation test: adherence by readmission (p = {pval:.3f})')
ax.set_xlabel('mean difference (readmit - no)')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_perm_adherence.png', dpi=120)

print('observed diff:', round(obs, 4), '| p-value:', round(pval, 4))
